# CollabMind: reranker + claim verifier training
Settings: **Accelerator = GPU T4 x2**, **Internet = On**. Then *Run All*. Expect roughly 1 to 2 hours.

Built on public models and data, credited in the project report: `cross-encoder/ms-marco-MiniLM-L-6-v2` (Apache-2.0), `microsoft/deberta-v3-small` (MIT), `cross-encoder/nli-deberta-v3-small` (Apache-2.0), `BAAI/bge-small-en-v1.5` (MIT), BEIR SciFact (CC BY-NC 4.0), RAGTruth (MIT), MNLI.

In [ ]:
!pip -q install sentence-transformers httpx pyarrow sentencepiece
import torch; print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")

## 1. Write the project code

In [ ]:
import json, os
from pathlib import Path
FILES = json.loads("{\"ml/__init__.py\": \"\", \"ml/data/__init__.py\": \"\", \"ml/data/hf_rows.py\": \"\\\"\\\"\\\"\\nml/data/hf_rows.py \\u2014 read public Hugging Face datasets over the datasets-server HTTP API.\\n\\nNo `datasets` or `torch` install is needed, so data preparation runs on any laptop. Pages are cached on\\ndisk (ml/data/cache) so a re-run costs nothing. Only public datasets are read; every dataset used is\\nlisted with its licence in ai/THIRD_PARTY_NOTICES.md.\\n\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nimport json\\nimport time\\nfrom pathlib import Path\\nfrom typing import Iterator, Optional\\n\\nimport httpx\\n\\nAPI = \\\"https://datasets-server.huggingface.co/rows\\\"\\nPAGE = 100                       # the API's maximum page size\\nPAUSE_SECONDS = 1.0              # between uncached requests\\nCACHE = Path(__file__).parent / \\\"cache\\\"\\n\\n\\ndef _page(dataset: str, config: str, split: str, offset: int, length: int) -> dict:\\n    key = f\\\"{dataset.replace('/', '__')}__{config}__{split}__{offset}__{length}.json\\\"\\n    path = CACHE / key\\n    if path.exists():\\n        return json.loads(path.read_text(encoding=\\\"utf-8\\\"))\\n    params = {\\\"dataset\\\": dataset, \\\"config\\\": config, \\\"split\\\": split, \\\"offset\\\": offset, \\\"length\\\": length}\\n    for attempt in range(7):\\n        r = httpx.get(API, params=params, timeout=60)\\n        if r.status_code == 200:\\n            CACHE.mkdir(parents=True, exist_ok=True)\\n            path.write_text(r.text, encoding=\\\"utf-8\\\")\\n            time.sleep(PAUSE_SECONDS)       # stay under the public rate limit\\n            return r.json()\\n        if r.status_code in (429, 500, 502, 503, 504):\\n            time.sleep(min(60, 5 * 2 ** attempt))\\n            continue\\n        r.raise_for_status()\\n    raise RuntimeError(f\\\"{dataset}/{config}/{split} @ {offset}: gave up after retries\\\")\\n\\n\\ndef iter_parquet(dataset: str, config: str, split: str, *, columns: Optional[list[str]] = None,\\n                 every: int = 1) -> Iterator[dict]:\\n    \\\"\\\"\\\"Rows of a whole split from its auto-converted Parquet files (one download per file, no rate limit).\\n\\n    Needs `pyarrow` (preinstalled on Kaggle/Colab). `every=k` keeps every k-th row for cheap subsampling.\\\"\\\"\\\"\\n    import pyarrow.parquet as pq\\n\\n    CACHE.mkdir(parents=True, exist_ok=True)\\n    urls = httpx.get(f\\\"https://huggingface.co/api/datasets/{dataset}/parquet/{config}/{split}\\\", timeout=60,\\n                     follow_redirects=True).raise_for_status().json()\\n    n = 0\\n    for i, url in enumerate(urls):\\n        path = CACHE / f\\\"{dataset.replace('/', '__')}__{config}__{split}__{i}.parquet\\\"\\n        if not path.exists():\\n            tmp = path.with_suffix(\\\".part\\\")\\n            for attempt in range(4):\\n                try:\\n                    with httpx.stream(\\\"GET\\\", url, timeout=300, follow_redirects=True) as r, open(tmp, \\\"wb\\\") as f:\\n                        r.raise_for_status()\\n                        for chunk in r.iter_bytes(1 << 20):\\n                            f.write(chunk)\\n                    break\\n                except httpx.TransportError:\\n                    if attempt == 3:\\n                        raise\\n                    time.sleep(5 * (attempt + 1))\\n            tmp.replace(path)\\n        for batch in pq.ParquetFile(path).iter_batches(batch_size=2048, columns=columns):\\n            for row in batch.to_pylist():\\n                if n % every == 0:\\n                    yield row\\n                n += 1\\n\\n\\ndef iter_rows(dataset: str, config: str, split: str, *, limit: Optional[int] = None,\\n              start: int = 0) -> Iterator[dict]:\\n    \\\"\\\"\\\"Rows in order from `start`, at most `limit` of them.\\\"\\\"\\\"\\n    offset, seen = start, 0\\n    while limit is None or seen < limit:\\n        length = PAGE if limit is None else min(PAGE, limit - seen)\\n        data = _page(dataset, config, split, offset, length)\\n        rows = data[\\\"rows\\\"]\\n        if not rows:\\n            return\\n        for r in rows:\\n            yield r[\\\"row\\\"]\\n        seen += len(rows)\\n        offset += len(rows)\\n        if offset >= data[\\\"num_rows_total\\\"]:\\n            return\\n\", \"ml/data/scifact.py\": \"\\\"\\\"\\\"\\nml/data/scifact.py \\u2014 the BEIR SciFact retrieval benchmark (scientific claims -> abstracts).\\n\\nBEIR protocol: queries in the qrels `train` split are for tuning, queries in `test` are the held-out\\ntest set and are never used for training. Source: BeIR/scifact (CC BY-NC 4.0).\\n\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nfrom ml.data.hf_rows import iter_rows\\n\\n\\ndef load_corpus() -> dict[str, str]:\\n    \\\"\\\"\\\"doc_id -> 'title. text'.\\\"\\\"\\\"\\n    return {r[\\\"_id\\\"]: f'{r[\\\"title\\\"]}. {r[\\\"text\\\"]}'.strip(\\\". \\\")\\n            for r in iter_rows(\\\"BeIR/scifact\\\", \\\"corpus\\\", \\\"corpus\\\")}\\n\\n\\ndef load_queries() -> dict[str, str]:\\n    return {r[\\\"_id\\\"]: r[\\\"text\\\"] for r in iter_rows(\\\"BeIR/scifact\\\", \\\"queries\\\", \\\"queries\\\")}\\n\\n\\ndef load_qrels(split: str) -> dict[str, set[str]]:\\n    \\\"\\\"\\\"query_id -> relevant doc ids (score > 0). `split` is 'train' or 'test'.\\\"\\\"\\\"\\n    out: dict[str, set[str]] = {}\\n    for r in iter_rows(\\\"BeIR/scifact-qrels\\\", \\\"default\\\", split):\\n        if r[\\\"score\\\"] > 0:\\n            out.setdefault(str(r[\\\"query-id\\\"]), set()).add(str(r[\\\"corpus-id\\\"]))\\n    return out\\n\", \"ml/data/make_reranker_data.py\": \"\\\"\\\"\\\"\\nml/data/make_reranker_data.py \\u2014 build reranker training data from BEIR SciFact.\\n\\n    python -m ml.data.make_reranker_data\\n\\nWrites ml/data/out/{train,dev}.jsonl with {\\\"query\\\", \\\"passage\\\", \\\"label\\\"} rows (the format\\nml/train_reranker.py reads). Positives come from the qrels `train` split; hard negatives are the\\nhighest-ranked BM25 documents that are NOT marked relevant, which is exactly the kind of near-miss a\\nfirst-stage retriever returns and a reranker must learn to push down.\\n\\nThe qrels `test` split is never touched here: it is the held-out test set for ml/eval/run_retrieval_eval.py.\\nQueries (not rows) are split into train/dev, so no query appears in both.\\n\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport json\\nimport random\\nfrom pathlib import Path\\n\\nfrom ml.data import scifact\\nfrom ml.eval.bm25 import BM25\\n\\nOUT = Path(__file__).parent / \\\"out\\\"\\n\\n\\ndef build(neg_per_pos: int, pool: int, dev_share: float, seed: int) -> tuple[list[dict], list[dict]]:\\n    rng = random.Random(seed)\\n    corpus, queries, qrels = scifact.load_corpus(), scifact.load_queries(), scifact.load_qrels(\\\"train\\\")\\n    index = BM25(corpus)\\n    qids = sorted(qrels)\\n    rng.shuffle(qids)\\n    n_dev = max(1, int(len(qids) * dev_share))\\n    dev_ids = set(qids[:n_dev])\\n\\n    train, dev = [], []\\n    for qid in qids:\\n        relevant = qrels[qid]\\n        ranked = [d for d, _ in index.search(queries[qid], k=pool) if d not in relevant]\\n        rows = [{\\\"query\\\": queries[qid], \\\"passage\\\": corpus[d], \\\"label\\\": 1} for d in relevant if d in corpus]\\n        negatives = ranked[: neg_per_pos * max(1, len(relevant)) * 2]          # hardest first\\n        rng.shuffle(negatives)\\n        rows += [{\\\"query\\\": queries[qid], \\\"passage\\\": corpus[d], \\\"label\\\": 0}\\n                 for d in negatives[: neg_per_pos * max(1, len(relevant))]]\\n        (dev if qid in dev_ids else train).extend(rows)\\n    return train, dev\\n\\n\\ndef main() -> None:\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\\"--neg-per-pos\\\", type=int, default=4)\\n    ap.add_argument(\\\"--pool\\\", type=int, default=30, help=\\\"BM25 depth negatives are drawn from\\\")\\n    ap.add_argument(\\\"--dev-share\\\", type=float, default=0.1)\\n    ap.add_argument(\\\"--seed\\\", type=int, default=13)\\n    args = ap.parse_args()\\n\\n    train, dev = build(args.neg_per_pos, args.pool, args.dev_share, args.seed)\\n    OUT.mkdir(parents=True, exist_ok=True)\\n    for name, rows in ((\\\"train\\\", train), (\\\"dev\\\", dev)):\\n        (OUT / f\\\"{name}.jsonl\\\").write_text(\\\"\\\\n\\\".join(json.dumps(r) for r in rows) + \\\"\\\\n\\\", encoding=\\\"utf-8\\\")\\n        pos = sum(r[\\\"label\\\"] for r in rows)\\n        print(f\\\"{name}: {len(rows)} rows ({pos} positive, {len(rows) - pos} hard negative) -> {OUT / (name + '.jsonl')}\\\")\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    main()\\n\", \"ml/eval/__init__.py\": \"\", \"ml/eval/bm25.py\": \"\\\"\\\"\\\"\\nml/eval/bm25.py \\u2014 a small dependency-free BM25 (Okapi) index.\\n\\nUsed as the lexical first stage in the retrieval ablation and to mine hard negatives for reranker\\ntraining. Mirrors the keyword half of the platform's hybrid retrieval closely enough for a benchmark.\\n\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nimport math\\nimport re\\nfrom collections import Counter\\n\\n_TOKEN = re.compile(r\\\"[a-z0-9]+\\\")\\n_STOP = frozenset(\\\"a an and are as at be by for from has have in is it its of on or that the to was were with\\\".split())\\n\\n\\ndef tokenize(text: str) -> list[str]:\\n    return [t for t in _TOKEN.findall(text.lower()) if t not in _STOP]\\n\\n\\nclass BM25:\\n    def __init__(self, docs: dict[str, str], k1: float = 1.5, b: float = 0.75):\\n        self.k1, self.b = k1, b\\n        self.ids = list(docs)\\n        self.tf = [Counter(tokenize(docs[i])) for i in self.ids]\\n        self.len = [sum(c.values()) for c in self.tf]\\n        self.avg = sum(self.len) / max(1, len(self.len))\\n        df: Counter = Counter()\\n        for c in self.tf:\\n            df.update(c.keys())\\n        n = len(self.ids)\\n        self.idf = {t: math.log(1 + (n - d + 0.5) / (d + 0.5)) for t, d in df.items()}\\n        self.postings: dict[str, list[int]] = {}\\n        for pos, c in enumerate(self.tf):\\n            for t in c:\\n                self.postings.setdefault(t, []).append(pos)\\n\\n    def search(self, query: str, k: int = 100) -> list[tuple[str, float]]:\\n        scores: dict[int, float] = {}\\n        for t in set(tokenize(query)):\\n            idf = self.idf.get(t)\\n            if idf is None:\\n                continue\\n            for pos in self.postings[t]:\\n                f = self.tf[pos][t]\\n                norm = f + self.k1 * (1 - self.b + self.b * self.len[pos] / self.avg)\\n                scores[pos] = scores.get(pos, 0.0) + idf * f * (self.k1 + 1) / norm\\n        top = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)[:k]\\n        return [(self.ids[p], s) for p, s in top]\\n\", \"ml/eval/metrics.py\": \"\\\"\\\"\\\"\\nml/eval/metrics.py \\u2014 retrieval metrics for the ablation table (Recall@k, MRR@10, NDCG@10).\\n\\nPure functions over one query's ranked result list. `ranked` is a list of chunk keys in rank order;\\n`relevant` is the set of keys a human marked relevant for that query (binary relevance).\\n\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nimport math\\nfrom typing import Hashable, Iterable, Sequence\\n\\n\\ndef recall_at_k(ranked: Sequence[Hashable], relevant: set, k: int) -> float:\\n    \\\"\\\"\\\"Share of the relevant chunks found in the top k.\\\"\\\"\\\"\\n    if not relevant:\\n        raise ValueError(\\\"a query needs at least one relevant chunk\\\")\\n    return len(set(ranked[:k]) & relevant) / len(relevant)\\n\\n\\ndef mrr_at_k(ranked: Sequence[Hashable], relevant: set, k: int = 10) -> float:\\n    \\\"\\\"\\\"1 / rank of the first relevant chunk in the top k, else 0.\\\"\\\"\\\"\\n    for i, key in enumerate(ranked[:k], start=1):\\n        if key in relevant:\\n            return 1.0 / i\\n    return 0.0\\n\\n\\ndef ndcg_at_k(ranked: Sequence[Hashable], relevant: set, k: int = 10) -> float:\\n    \\\"\\\"\\\"Binary-relevance NDCG: DCG of the ranking over the DCG of a perfect ranking.\\\"\\\"\\\"\\n    if not relevant:\\n        raise ValueError(\\\"a query needs at least one relevant chunk\\\")\\n    dcg = sum(1.0 / math.log2(i + 1) for i, key in enumerate(ranked[:k], start=1) if key in relevant)\\n    ideal = sum(1.0 / math.log2(i + 1) for i in range(1, min(len(relevant), k) + 1))\\n    return dcg / ideal\\n\\n\\ndef evaluate(runs: Iterable[tuple[Sequence[Hashable], set]], ks: Sequence[int] = (1, 5, 10)) -> dict[str, float]:\\n    \\\"\\\"\\\"Mean of every metric over queries: [(ranked, relevant), ...].\\\"\\\"\\\"\\n    runs = list(runs)\\n    if not runs:\\n        raise ValueError(\\\"no queries to evaluate\\\")\\n    out: dict[str, float] = {}\\n    for k in ks:\\n        out[f\\\"recall@{k}\\\"] = sum(recall_at_k(r, rel, k) for r, rel in runs) / len(runs)\\n    out[\\\"mrr@10\\\"] = sum(mrr_at_k(r, rel, 10) for r, rel in runs) / len(runs)\\n    out[\\\"ndcg@10\\\"] = sum(ndcg_at_k(r, rel, 10) for r, rel in runs) / len(runs)\\n    out[\\\"queries\\\"] = float(len(runs))\\n    return out\\n\", \"ml/eval/classification.py\": \"\\\"\\\"\\\"\\nml/eval/classification.py \\u2014 binary-classification metrics for the verifier and the calibration model.\\n\\nPure Python (no numpy/sklearn) so tests run anywhere. Convention: label 1 = positive class; `scores` are\\nthe model's probability (or any monotone score) of the positive class.\\n\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nfrom typing import Sequence\\n\\n\\ndef confusion(labels: Sequence[int], preds: Sequence[int]) -> dict[str, int]:\\n    tp = sum(1 for y, p in zip(labels, preds) if y == 1 and p == 1)\\n    fp = sum(1 for y, p in zip(labels, preds) if y == 0 and p == 1)\\n    fn = sum(1 for y, p in zip(labels, preds) if y == 1 and p == 0)\\n    tn = sum(1 for y, p in zip(labels, preds) if y == 0 and p == 0)\\n    return {\\\"tp\\\": tp, \\\"fp\\\": fp, \\\"fn\\\": fn, \\\"tn\\\": tn}\\n\\n\\ndef prf(labels: Sequence[int], preds: Sequence[int]) -> dict[str, float]:\\n    c = confusion(labels, preds)\\n    precision = c[\\\"tp\\\"] / (c[\\\"tp\\\"] + c[\\\"fp\\\"]) if c[\\\"tp\\\"] + c[\\\"fp\\\"] else 0.0\\n    recall = c[\\\"tp\\\"] / (c[\\\"tp\\\"] + c[\\\"fn\\\"]) if c[\\\"tp\\\"] + c[\\\"fn\\\"] else 0.0\\n    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0\\n    return {\\\"precision\\\": precision, \\\"recall\\\": recall, \\\"f1\\\": f1, **{k: float(v) for k, v in c.items()}}\\n\\n\\ndef roc_auc(labels: Sequence[int], scores: Sequence[float]) -> float:\\n    \\\"\\\"\\\"Area under the ROC curve via the rank-sum (Mann-Whitney) formula; ties get average rank.\\\"\\\"\\\"\\n    pos = sum(labels)\\n    neg = len(labels) - pos\\n    if pos == 0 or neg == 0:\\n        raise ValueError(\\\"AUC needs both classes\\\")\\n    order = sorted(range(len(scores)), key=lambda i: scores[i])\\n    ranks = [0.0] * len(scores)\\n    i = 0\\n    while i < len(order):\\n        j = i\\n        while j + 1 < len(order) and scores[order[j + 1]] == scores[order[i]]:\\n            j += 1\\n        for k in range(i, j + 1):\\n            ranks[order[k]] = (i + j) / 2 + 1\\n        i = j + 1\\n    rank_sum = sum(r for r, y in zip(ranks, labels) if y == 1)\\n    return (rank_sum - pos * (pos + 1) / 2) / (pos * neg)\\n\\n\\ndef best_threshold(labels: Sequence[int], scores: Sequence[float]) -> float:\\n    \\\"\\\"\\\"Threshold maximising F1 (pick it on a DEV set, then apply it unchanged to the test set).\\\"\\\"\\\"\\n    best_t, best_f1 = 0.5, -1.0\\n    for t in sorted(set(scores)):\\n        f1 = prf(labels, [1 if s >= t else 0 for s in scores])[\\\"f1\\\"]\\n        if f1 > best_f1:\\n            best_t, best_f1 = t, f1\\n    return best_t\\n\\n\\ndef ece(labels: Sequence[int], probs: Sequence[float], bins: int = 10) -> float:\\n    \\\"\\\"\\\"Expected calibration error of positive-class probabilities (equal-width bins).\\\"\\\"\\\"\\n    n = len(labels)\\n    total = 0.0\\n    for b in range(bins):\\n        lo, hi = b / bins, (b + 1) / bins\\n        idx = [i for i, p in enumerate(probs) if (lo <= p < hi) or (b == bins - 1 and p == 1.0)]\\n        if idx:\\n            conf = sum(probs[i] for i in idx) / len(idx)\\n            acc = sum(labels[i] for i in idx) / len(idx)\\n            total += len(idx) / n * abs(acc - conf)\\n    return total\\n\", \"ml/eval/run_retrieval_eval.py\": \"\\\"\\\"\\\"\\nml/eval/run_retrieval_eval.py \\u2014 the retrieval ablation table on the held-out BEIR SciFact test queries.\\n\\n    python -m ml.eval.run_retrieval_eval                                   # BM25 only (CPU, no torch)\\n    python -m ml.eval.run_retrieval_eval --dense BAAI/bge-small-en-v1.5 \\\\\\n        --rerank cross-encoder/ms-marco-MiniLM-L-6-v2 --rerank models/reranker-v1\\n\\nSystems reported (each a row; every number is on the SAME 300 test queries):\\n    bm25                      lexical first stage\\n    dense:<model>             bi-encoder first stage (needs sentence-transformers)\\n    hybrid:<model>            BM25 + dense fused with Reciprocal Rank Fusion (like the platform's hybrid search)\\n    <first stage>+rerank:<m>  cross-encoder re-scores the top --candidates of that first stage\\nMetrics: Recall@1/5/10, MRR@10, NDCG@10, and mean latency per query in ms (reranker stage only for rerank rows).\\nWrites ml/data/out/retrieval_results.{json,md}.\\n\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport json\\nimport time\\nfrom pathlib import Path\\n\\nfrom ml.data import scifact\\nfrom ml.eval.bm25 import BM25\\nfrom ml.eval.metrics import evaluate\\n\\nOUT = Path(__file__).parent.parent / \\\"data\\\" / \\\"out\\\"\\nRRF_K = 60\\n\\n\\ndef rrf(*rankings: list[str], k: int = RRF_K) -> list[str]:\\n    score: dict[str, float] = {}\\n    for ranking in rankings:\\n        for rank, doc in enumerate(ranking, start=1):\\n            score[doc] = score.get(doc, 0.0) + 1.0 / (k + rank)\\n    return sorted(score, key=score.get, reverse=True)\\n\\n\\ndef main() -> None:\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\\"--dense\\\", help=\\\"sentence-transformers bi-encoder id/path (optional)\\\")\\n    ap.add_argument(\\\"--rerank\\\", action=\\\"append\\\", default=[], help=\\\"cross-encoder id/path; repeatable\\\")\\n    ap.add_argument(\\\"--candidates\\\", type=int, default=50, help=\\\"first-stage depth given to the reranker\\\")\\n    ap.add_argument(\\\"--limit\\\", type=int, default=0, help=\\\"evaluate only the first N test queries (smoke test)\\\")\\n    args = ap.parse_args()\\n\\n    corpus, queries, qrels = scifact.load_corpus(), scifact.load_queries(), scifact.load_qrels(\\\"test\\\")\\n    qids = sorted(qrels)[: args.limit or None]\\n    bm25 = BM25(corpus)\\n    depth = max(100, args.candidates)\\n\\n    first_stage: dict[str, dict[str, list[str]]] = {\\\"bm25\\\": {q: [d for d, _ in bm25.search(queries[q], depth)] for q in qids}}\\n    if args.dense:\\n        from sentence_transformers import SentenceTransformer\\n\\n        enc = SentenceTransformer(args.dense)\\n        ids = list(corpus)\\n        doc_vecs = enc.encode([corpus[i] for i in ids], batch_size=64, normalize_embeddings=True, show_progress_bar=False)\\n        q_vecs = enc.encode([queries[q] for q in qids], batch_size=64, normalize_embeddings=True, show_progress_bar=False)\\n        sims = q_vecs @ doc_vecs.T\\n        dense = {q: [ids[j] for j in sims[n].argsort()[::-1][:depth]] for n, q in enumerate(qids)}\\n        first_stage[f\\\"dense:{args.dense}\\\"] = dense\\n        first_stage[f\\\"hybrid:{args.dense}\\\"] = {q: rrf(first_stage[\\\"bm25\\\"][q], dense[q])[:depth] for q in qids}\\n\\n    results: list[dict] = []\\n\\n    def record(name: str, ranked_by_q: dict[str, list[str]], latency_ms: float = 0.0) -> None:\\n        metrics = evaluate([(ranked_by_q[q], qrels[q]) for q in qids])\\n        results.append({\\\"system\\\": name, **metrics, \\\"latency_ms\\\": round(latency_ms, 1)})\\n        print(f\\\"{name:<60} MRR@10={metrics['mrr@10']:.3f} NDCG@10={metrics['ndcg@10']:.3f} R@10={metrics['recall@10']:.3f}\\\")\\n\\n    for name, ranked in first_stage.items():\\n        record(name, ranked)\\n\\n    for model_name in args.rerank:\\n        from sentence_transformers import CrossEncoder\\n\\n        ce = CrossEncoder(model_name, max_length=512)\\n        for stage, ranked in first_stage.items():\\n            reranked, spent = {}, 0.0\\n            for q in qids:\\n                cands = ranked[q][: args.candidates]\\n                t0 = time.perf_counter()\\n                scores = ce.predict([(queries[q], corpus[d]) for d in cands], batch_size=32, show_progress_bar=False)\\n                spent += time.perf_counter() - t0\\n                reranked[q] = [d for d, _ in sorted(zip(cands, scores), key=lambda x: x[1], reverse=True)]\\n            record(f\\\"{stage}+rerank:{model_name}\\\", reranked, 1000 * spent / len(qids))\\n\\n    OUT.mkdir(parents=True, exist_ok=True)\\n    (OUT / \\\"retrieval_results.json\\\").write_text(json.dumps(results, indent=2), encoding=\\\"utf-8\\\")\\n    cols = [\\\"recall@1\\\", \\\"recall@5\\\", \\\"recall@10\\\", \\\"mrr@10\\\", \\\"ndcg@10\\\", \\\"latency_ms\\\"]\\n    lines = [\\\"| system | \\\" + \\\" | \\\".join(cols) + \\\" |\\\", \\\"|---|\\\" + \\\"---|\\\" * len(cols)]\\n    for r in results:\\n        lines.append(f\\\"| {r['system']} | \\\" + \\\" | \\\".join(f\\\"{r[c]:.3f}\\\" if c != \\\"latency_ms\\\" else f\\\"{r[c]:.0f}\\\" for c in cols) + \\\" |\\\")\\n    (OUT / \\\"retrieval_results.md\\\").write_text(\\\"\\\\n\\\".join(lines) + f\\\"\\\\n\\\\nTest queries: {len(qids)} (BEIR SciFact test)\\\\n\\\", encoding=\\\"utf-8\\\")\\n    print(\\\"\\\\n\\\" + \\\"\\\\n\\\".join(lines))\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    main()\\n\", \"ml/cross_encoder_train.py\": \"\\\"\\\"\\\"\\nml/cross_encoder_train.py \\u2014 a plain PyTorch + Transformers training loop for single-logit cross-encoders.\\n\\nShared by the reranker (ml/train_reranker.py) and the verifier (ml/verifier/train_verifier.py). Written\\ndirectly against torch/transformers (no sentence-transformers training API), so it does not depend on that\\nlibrary's version and every step of fine-tuning is visible:\\n\\n    tokenise (text_a, text_b) pairs  ->  AutoModelForSequenceClassification(num_labels=1)\\n    -> logits -> BCE-with-logits loss -> AdamW + linear warmup/decay -> mixed precision (fp16) on GPU\\n    -> after each epoch run `dev_metric`; keep the best epoch's weights.\\n\\nThe saved folder is a normal Hugging Face model directory, loadable by sentence-transformers' CrossEncoder\\n(which rag/reranker.py and rag/verifier.py use at serving time).\\n\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nimport json\\nimport math\\nimport random\\nimport time\\nfrom pathlib import Path\\nfrom typing import Callable, Sequence\\n\\nPair = tuple[str, str]\\nPredict = Callable[[Sequence[Pair]], list[float]]       # (text_a, text_b) pairs -> raw logits\\n\\n\\ndef make_predict(model, tokenizer, device, max_length: int, batch_size: int = 64) -> Predict:\\n    import torch\\n\\n    def predict(pairs: Sequence[Pair]) -> list[float]:\\n        was_training = model.training\\n        model.eval()\\n        out: list[float] = []\\n        order = sorted(range(len(pairs)), key=lambda i: len(pairs[i][0]) + len(pairs[i][1]))   # less padding\\n        with torch.no_grad():\\n            for s in range(0, len(order), batch_size):\\n                idx = order[s:s + batch_size]\\n                enc = tokenizer([pairs[i][0] for i in idx], [pairs[i][1] for i in idx], truncation=\\\"longest_first\\\",\\n                                max_length=max_length, padding=True, return_tensors=\\\"pt\\\").to(device)\\n                logits = model(**enc).logits.view(-1).float().cpu().tolist()\\n                out.extend(zip(idx, logits))\\n        model.train(was_training)\\n        scores = [0.0] * len(pairs)\\n        for i, v in out:\\n            scores[i] = v\\n        return scores\\n\\n    return predict\\n\\n\\ndef train_cross_encoder(\\n    base: str,\\n    pairs: Sequence[Pair],\\n    labels: Sequence[int],\\n    out_dir: str,\\n    dev_metric: Callable[[Predict], tuple[float, dict]],\\n    *,\\n    epochs: int = 2,\\n    batch_size: int = 16,\\n    lr: float = 2e-5,\\n    max_length: int = 384,\\n    warmup_share: float = 0.1,\\n    weight_decay: float = 0.01,\\n    seed: int = 13,\\n    fp16: bool = True,\\n) -> dict:\\n    \\\"\\\"\\\"Fine-tune `base`; returns {\\\"best_score\\\", \\\"best_epoch\\\", \\\"history\\\"} and writes the best model to out_dir.\\n\\n    `dev_metric(predict)` returns (score to maximise, details to log); it is the ONLY place dev data is used.\\n    \\\"\\\"\\\"\\n    import torch\\n    from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup\\n\\n    random.seed(seed)\\n    torch.manual_seed(seed)\\n    device = torch.device(\\\"cuda\\\" if torch.cuda.is_available() else \\\"cpu\\\")\\n    use_amp = fp16 and device.type == \\\"cuda\\\"\\n    tokenizer = AutoTokenizer.from_pretrained(base)\\n    model = AutoModelForSequenceClassification.from_pretrained(base, num_labels=1, ignore_mismatched_sizes=True).to(device)\\n    predict = make_predict(model, tokenizer, device, max_length)\\n\\n    steps_per_epoch = math.ceil(len(pairs) / batch_size)\\n    total = steps_per_epoch * epochs\\n    no_decay = (\\\"bias\\\", \\\"LayerNorm.weight\\\", \\\"layernorm\\\", \\\"layer_norm\\\")\\n    params = [\\n        {\\\"params\\\": [p for n, p in model.named_parameters() if not any(k in n for k in no_decay)], \\\"weight_decay\\\": weight_decay},\\n        {\\\"params\\\": [p for n, p in model.named_parameters() if any(k in n for k in no_decay)], \\\"weight_decay\\\": 0.0},\\n    ]\\n    optimizer = torch.optim.AdamW(params, lr=lr)\\n    scheduler = get_linear_schedule_with_warmup(optimizer, max(1, int(warmup_share * total)), total)\\n    scaler = torch.amp.GradScaler(\\\"cuda\\\", enabled=use_amp)\\n    loss_fn = torch.nn.BCEWithLogitsLoss()\\n\\n    best, best_epoch, history = -math.inf, 0, []\\n    print(f\\\"training {base} on {len(pairs)} pairs, {epochs} epochs, device={device.type}, fp16={use_amp}\\\", flush=True)\\n    for epoch in range(1, epochs + 1):\\n        order = list(range(len(pairs)))\\n        random.shuffle(order)\\n        model.train()\\n        running, t0 = 0.0, time.time()\\n        for step, s in enumerate(range(0, len(order), batch_size), start=1):\\n            idx = order[s:s + batch_size]\\n            enc = tokenizer([pairs[i][0] for i in idx], [pairs[i][1] for i in idx], truncation=\\\"longest_first\\\",\\n                            max_length=max_length, padding=True, return_tensors=\\\"pt\\\").to(device)\\n            y = torch.tensor([float(labels[i]) for i in idx], device=device)\\n            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):\\n                logits = model(**enc).logits.view(-1)\\n            loss = loss_fn(logits.float(), y)\\n            optimizer.zero_grad(set_to_none=True)\\n            scaler.scale(loss).backward()\\n            scaler.unscale_(optimizer)\\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\\n            scaler.step(optimizer)\\n            scaler.update()\\n            scheduler.step()\\n            running += loss.item()\\n            if step % 50 == 0 or step == steps_per_epoch:\\n                print(f\\\"epoch {epoch} step {step}/{steps_per_epoch} loss={running / step:.4f} \\\"\\n                      f\\\"({time.time() - t0:.0f}s)\\\", flush=True)\\n        score, details = dev_metric(predict)\\n        history.append({\\\"epoch\\\": epoch, \\\"train_loss\\\": running / steps_per_epoch, \\\"dev_score\\\": score, **details})\\n        print(f\\\"epoch {epoch} dev_score={score:.4f} {details}\\\", flush=True)\\n        if score > best:\\n            best, best_epoch = score, epoch\\n            Path(out_dir).mkdir(parents=True, exist_ok=True)\\n            model.save_pretrained(out_dir)\\n            tokenizer.save_pretrained(out_dir)\\n    Path(out_dir, \\\"training_history.json\\\").write_text(json.dumps(history, indent=2), encoding=\\\"utf-8\\\")\\n    print(f\\\"best epoch {best_epoch} (dev_score={best:.4f}) saved to {out_dir}\\\", flush=True)\\n    return {\\\"best_score\\\": best, \\\"best_epoch\\\": best_epoch, \\\"history\\\": history}\\n\", \"ml/train_reranker.py\": \"\\\"\\\"\\\"\\nml/train_reranker.py \\u2014 fine-tune a cross-encoder reranker (run on a Kaggle/Colab GPU, not in the service).\\n\\nInput: a JSONL file with one example per line:\\n    {\\\"query\\\": \\\"...\\\", \\\"passage\\\": \\\"...\\\", \\\"label\\\": 1}      # 1 = relevant, 0 = hard negative\\nHard negatives come from the CURRENT first-stage retriever (ml/data/make_reranker_data.py mines them with BM25).\\n\\n    pip install -r requirements-ml.txt\\n    python -m ml.train_reranker --train ml/data/out/train.jsonl --dev ml/data/out/dev.jsonl \\\\\\n        --base cross-encoder/ms-marco-MiniLM-L-6-v2 --out models/reranker-v1\\n\\nLoss: binary cross-entropy on one relevance logit per (query, passage). The best epoch is chosen by\\ndev MRR@10 (queries are ranked among their own positives and negatives). The held-out TEST queries are\\nnever passed here. The output directory is what RERANKER_MODEL points at.\\n\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport json\\nfrom pathlib import Path\\n\\nfrom ml.cross_encoder_train import train_cross_encoder\\nfrom ml.eval.metrics import mrr_at_k\\n\\n\\ndef read_jsonl(path: str) -> list[dict]:\\n    rows = []\\n    for n, line in enumerate(Path(path).read_text(encoding=\\\"utf-8\\\").splitlines(), start=1):\\n        if not line.strip():\\n            continue\\n        row = json.loads(line)\\n        if not {\\\"query\\\", \\\"passage\\\", \\\"label\\\"} <= row.keys() or row[\\\"label\\\"] not in (0, 1):\\n            raise ValueError(f\\\"{path}:{n}: need query, passage and a 0/1 label\\\")\\n        rows.append(row)\\n    if not rows:\\n        raise ValueError(f\\\"{path} has no examples\\\")\\n    return rows\\n\\n\\ndef dev_mrr(dev: list[dict]):\\n    \\\"\\\"\\\"MRR@10 of ranking each dev query's candidates by the model's score.\\\"\\\"\\\"\\n    groups: dict[str, list[dict]] = {}\\n    for r in dev:\\n        groups.setdefault(r[\\\"query\\\"], []).append(r)\\n    groups = {q: g for q, g in groups.items() if any(r[\\\"label\\\"] for r in g) and any(not r[\\\"label\\\"] for r in g)}\\n    flat = [(q, r[\\\"passage\\\"]) for q, g in groups.items() for r in g]\\n\\n    def metric(predict):\\n        scores = iter(predict(flat))\\n        total = 0.0\\n        for q, g in groups.items():\\n            scored = [(next(scores), i) for i in range(len(g))]\\n            ranked = [i for _, i in sorted(scored, reverse=True)]\\n            total += mrr_at_k(ranked, {i for i, r in enumerate(g) if r[\\\"label\\\"]}, 10)\\n        mrr = total / max(1, len(groups))\\n        return mrr, {\\\"dev_mrr@10\\\": round(mrr, 4), \\\"dev_queries\\\": len(groups)}\\n\\n    return metric\\n\\n\\ndef main() -> None:\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\\"--train\\\", required=True)\\n    ap.add_argument(\\\"--dev\\\", required=True)\\n    ap.add_argument(\\\"--base\\\", default=\\\"cross-encoder/ms-marco-MiniLM-L-6-v2\\\")\\n    ap.add_argument(\\\"--out\\\", required=True)\\n    ap.add_argument(\\\"--epochs\\\", type=int, default=2)\\n    ap.add_argument(\\\"--batch-size\\\", type=int, default=16)\\n    ap.add_argument(\\\"--lr\\\", type=float, default=2e-5)\\n    ap.add_argument(\\\"--max-length\\\", type=int, default=384)\\n    ap.add_argument(\\\"--seed\\\", type=int, default=13)\\n    ap.add_argument(\\\"--no-fp16\\\", action=\\\"store_true\\\")\\n    args = ap.parse_args()\\n\\n    train, dev = read_jsonl(args.train), read_jsonl(args.dev)\\n    train_cross_encoder(\\n        args.base, [(r[\\\"query\\\"], r[\\\"passage\\\"]) for r in train], [r[\\\"label\\\"] for r in train], args.out,\\n        dev_mrr(dev), epochs=args.epochs, batch_size=args.batch_size, lr=args.lr,\\n        max_length=args.max_length, seed=args.seed, fp16=not args.no_fp16)\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    main()\\n\", \"ml/verifier/__init__.py\": \"\", \"ml/verifier/sentences.py\": \"\\\"\\\"\\\"\\nml/verifier/sentences.py \\u2014 claim splitting and premise selection shared by training, evaluation and the\\nservice (rag/verifier.py), so the model sees the same kind of input everywhere.\\n\\nA \\\"claim\\\" is one sentence of an answer with citation markers like [1] removed. The \\\"premise\\\" for a claim is\\nthe few passage windows that best overlap it lexically (cross-encoders read ~512 tokens, so a long context\\ncannot be fed whole).\\n\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nimport re\\n\\n_CITE = re.compile(r\\\"\\\\s*\\\\[\\\\d+(?:\\\\s*,\\\\s*\\\\d+)*\\\\]\\\")\\n_SPLIT = re.compile(r\\\"(?<=[.!?])\\\\s+(?=[A-Z0-9\\\\\\\"'(\\\\[])|\\\\n+\\\")\\n_WORD = re.compile(r\\\"[a-z0-9]+\\\")\\n_STOP = frozenset(\\\"a an and are as at be by for from has have in is it its of on or that the to was were with\\\".split())\\nMIN_CLAIM_CHARS = 20\\nMAX_PREMISE_CHARS = 1400\\n\\n\\ndef strip_citations(text: str) -> str:\\n    return _CITE.sub(\\\"\\\", text)\\n\\n\\ndef split_sentences(text: str) -> list[str]:\\n    return [s.strip() for s in _SPLIT.split(text) if s and s.strip()]\\n\\n\\ndef sentence_spans(text: str) -> list[tuple[int, int, str]]:\\n    \\\"\\\"\\\"(start, end, sentence) with character offsets into `text`; used to map span-level labels to sentences.\\\"\\\"\\\"\\n    spans, pos = [], 0\\n    for m in list(_SPLIT.finditer(text)) + [None]:\\n        end = m.start() if m else len(text)\\n        seg = text[pos:end]\\n        if seg.strip():\\n            lead = len(seg) - len(seg.lstrip())\\n            spans.append((pos + lead, pos + len(seg.rstrip()), seg.strip()))\\n        pos = m.end() if m else len(text)\\n    return spans\\n\\n\\ndef split_claims(answer: str) -> list[str]:\\n    \\\"\\\"\\\"Checkable sentences of an answer: citations removed, headings/fragments dropped.\\\"\\\"\\\"\\n    claims = []\\n    for s in split_sentences(strip_citations(answer)):\\n        s = s.lstrip(\\\"-*\\u2022# \\\").strip()\\n        if len(s) >= MIN_CLAIM_CHARS and len(_WORD.findall(s.lower())) >= 4:\\n            claims.append(s)\\n    return claims\\n\\n\\ndef _terms(text: str) -> set[str]:\\n    return {w for w in _WORD.findall(text.lower()) if w not in _STOP}\\n\\n\\ndef select_premise(claim: str, context: str, max_chars: int = MAX_PREMISE_CHARS, window: int = 2) -> str:\\n    \\\"\\\"\\\"The context windows (`window` consecutive sentences) that overlap the claim most, in original order.\\\"\\\"\\\"\\n    if len(context) <= max_chars:\\n        return context\\n    sents = split_sentences(context) or [context]\\n    wins = [(i, \\\" \\\".join(sents[i:i + window])) for i in range(0, len(sents), max(1, window - 1))]\\n    want = _terms(claim)\\n    ranked = sorted(wins, key=lambda w: len(want & _terms(w[1])) / (1 + len(_terms(w[1])) ** 0.5), reverse=True)\\n    chosen, used = [], 0\\n    for i, text in ranked:\\n        if used + len(text) > max_chars and chosen:\\n            break\\n        chosen.append((i, text[:max_chars]))\\n        used += len(text)\\n    return \\\" \\\".join(t for _, t in sorted(chosen))\\n\", \"ml/verifier/make_data.py\": \"\\\"\\\"\\\"\\nml/verifier/make_data.py \\u2014 build the claim-support dataset for the verifier.\\n\\n    python -m ml.verifier.make_data\\n\\nEach row: {\\\"premise\\\", \\\"hypothesis\\\", \\\"label\\\", \\\"source\\\"} with label 1 = SUPPORTED by the premise, 0 = NOT.\\n\\n  * RAGTruth (wandb/RAGTruth-processed, MIT): real RAG answers from several LLMs with human span-level\\n    hallucination annotations. Each answer sentence becomes a claim; it is unsupported (0) if it overlaps an\\n    annotated hallucination span. Premise = the context windows closest to the claim (select_premise), the same\\n    selection the service uses. Only QA and Summary tasks (Data2txt contexts are JSON, not prose).\\n    Split by RESPONSE: 90% train / 10% dev from RAGTruth `train`; RAGTruth `test` is the held-out test set.\\n  * MNLI (nyu-mll/multi_nli): a small slice, entailment -> 1, neutral/contradiction -> 0, as general\\n    entailment signal for training only (never evaluated on).\\n\\nWrites ml/data/out/verifier_{train,dev,test}.jsonl.\\n\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport hashlib\\nimport json\\nfrom pathlib import Path\\n\\nfrom ml.data.hf_rows import iter_parquet\\nfrom ml.verifier.sentences import MIN_CLAIM_CHARS, select_premise, sentence_spans, strip_citations\\n\\nOUT = Path(__file__).parent.parent / \\\"data\\\" / \\\"out\\\"\\nTASKS = {\\\"QA\\\", \\\"Summary\\\"}\\n\\n\\ndef ragtruth_claims(row: dict) -> list[dict]:\\n    output = row[\\\"output\\\"]\\n    try:\\n        spans = [(int(s[\\\"start\\\"]), int(s[\\\"end\\\"])) for s in json.loads(row[\\\"hallucination_labels\\\"] or \\\"[]\\\")]\\n    except (ValueError, KeyError, TypeError):\\n        return []\\n    out = []\\n    for start, end, sent in sentence_spans(output):\\n        if len(sent) < MIN_CLAIM_CHARS:\\n            continue\\n        bad = any(start < e and s < end for s, e in spans)\\n        out.append({\\\"premise\\\": select_premise(sent, row[\\\"context\\\"]), \\\"hypothesis\\\": strip_citations(sent),\\n                    \\\"label\\\": 0 if bad else 1, \\\"source\\\": f\\\"ragtruth:{row['task_type']}\\\"})\\n    return out\\n\\n\\ndef _is_dev(response_id: str) -> bool:\\n    return int(hashlib.md5(response_id.encode()).hexdigest(), 16) % 10 == 0\\n\\n\\ndef build_ragtruth() -> tuple[list[dict], list[dict], list[dict]]:\\n    train, dev, test = [], [], []\\n    cols = [\\\"id\\\", \\\"context\\\", \\\"output\\\", \\\"task_type\\\", \\\"hallucination_labels\\\"]\\n    for r in iter_parquet(\\\"wandb/RAGTruth-processed\\\", \\\"default\\\", \\\"train\\\", columns=cols):\\n        if r[\\\"task_type\\\"] in TASKS:\\n            (dev if _is_dev(r[\\\"id\\\"]) else train).extend(ragtruth_claims(r))\\n    for r in iter_parquet(\\\"wandb/RAGTruth-processed\\\", \\\"default\\\", \\\"test\\\", columns=cols):\\n        if r[\\\"task_type\\\"] in TASKS:\\n            test.extend(ragtruth_claims(r))\\n    return train, dev, test\\n\\n\\ndef build_mnli(rows_wanted: int) -> list[dict]:\\n    every = max(1, 392_702 // max(1, rows_wanted))          # MNLI train has 392,702 pairs\\n    return [{\\\"premise\\\": r[\\\"premise\\\"], \\\"hypothesis\\\": r[\\\"hypothesis\\\"], \\\"label\\\": 1 if r[\\\"label\\\"] == 0 else 0,\\n             \\\"source\\\": \\\"mnli\\\"}\\n            for r in iter_parquet(\\\"nyu-mll/multi_nli\\\", \\\"default\\\", \\\"train\\\",\\n                                  columns=[\\\"premise\\\", \\\"hypothesis\\\", \\\"label\\\"], every=every)]\\n\\n\\ndef write(name: str, rows: list[dict]) -> None:\\n    OUT.mkdir(parents=True, exist_ok=True)\\n    (OUT / f\\\"verifier_{name}.jsonl\\\").write_text(\\\"\\\\n\\\".join(json.dumps(r) for r in rows) + \\\"\\\\n\\\", encoding=\\\"utf-8\\\")\\n    pos = sum(r[\\\"label\\\"] for r in rows)\\n    print(f\\\"{name}: {len(rows)} rows, {pos} supported / {len(rows) - pos} unsupported \\\"\\n          f\\\"({100 * (len(rows) - pos) / max(1, len(rows)):.1f}% unsupported)\\\")\\n\\n\\ndef main() -> None:\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\\"--mnli-rows\\\", type=int, default=0,\\n                    help=\\\"MNLI rows mixed into training (0 = none; downloads a 214 MB file, so use on a fast link)\\\")\\n    args = ap.parse_args()\\n    train, dev, test = build_ragtruth()\\n    if args.mnli_rows:\\n        train += build_mnli(args.mnli_rows)\\n    write(\\\"train\\\", train)\\n    write(\\\"dev\\\", dev)\\n    write(\\\"test\\\", test)\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    main()\\n\", \"ml/verifier/train_verifier.py\": \"\\\"\\\"\\\"\\nml/verifier/train_verifier.py \\u2014 fine-tune the claim-support verifier (run on a Kaggle/Colab GPU).\\n\\n    python -m ml.verifier.train_verifier --train ml/data/out/verifier_train.jsonl \\\\\\n        --dev ml/data/out/verifier_dev.jsonl --out models/verifier-v1\\n\\nModel: DeBERTa-v3-small as a cross-encoder with ONE logit trained with binary cross-entropy\\n(sigmoid(logit) = probability the claim is SUPPORTED by the premise). Input is (premise, claim), truncated\\nto --max-length tokens. The best epoch is chosen by dev ROC-AUC. The held-out TEST file\\n(verifier_test.jsonl) is never passed here. The output directory is what VERIFIER_MODEL points at.\\n\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport json\\nimport random\\nfrom pathlib import Path\\n\\nfrom ml.cross_encoder_train import train_cross_encoder\\nfrom ml.eval.classification import prf, roc_auc\\n\\n\\ndef read_jsonl(path: str) -> list[dict]:\\n    rows = [json.loads(line) for line in Path(path).read_text(encoding=\\\"utf-8\\\").splitlines() if line.strip()]\\n    for n, r in enumerate(rows, start=1):\\n        if not {\\\"premise\\\", \\\"hypothesis\\\", \\\"label\\\"} <= r.keys() or r[\\\"label\\\"] not in (0, 1):\\n            raise ValueError(f\\\"{path}:{n}: need premise, hypothesis and a 0/1 label\\\")\\n    if not rows:\\n        raise ValueError(f\\\"{path} has no examples\\\")\\n    return rows\\n\\n\\ndef dev_auc(dev: list[dict]):\\n    pairs = [(r[\\\"premise\\\"], r[\\\"hypothesis\\\"]) for r in dev]\\n    labels = [r[\\\"label\\\"] for r in dev]\\n\\n    def metric(predict):\\n        logits = predict(pairs)\\n        auc = roc_auc(labels, logits)                       # how well P(supported) separates the two classes\\n        m = prf([1 - y for y in labels], [1 if x < 0 else 0 for x in logits])   # unsupported = positive, thr 0.5\\n        return auc, {\\\"dev_auc\\\": round(auc, 4), \\\"dev_unsupported_f1@0.5\\\": round(m[\\\"f1\\\"], 4)}\\n\\n    return metric\\n\\n\\ndef main() -> None:\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\\"--train\\\", required=True)\\n    ap.add_argument(\\\"--dev\\\", required=True)\\n    ap.add_argument(\\\"--base\\\", default=\\\"microsoft/deberta-v3-small\\\")\\n    ap.add_argument(\\\"--out\\\", required=True)\\n    ap.add_argument(\\\"--epochs\\\", type=int, default=2)\\n    ap.add_argument(\\\"--batch-size\\\", type=int, default=16)\\n    ap.add_argument(\\\"--lr\\\", type=float, default=2e-5)\\n    ap.add_argument(\\\"--max-length\\\", type=int, default=384)\\n    ap.add_argument(\\\"--dev-size\\\", type=int, default=3000, help=\\\"dev rows used for the per-epoch check\\\")\\n    ap.add_argument(\\\"--seed\\\", type=int, default=13)\\n    ap.add_argument(\\\"--no-fp16\\\", action=\\\"store_true\\\")\\n    args = ap.parse_args()\\n\\n    train, dev = read_jsonl(args.train), read_jsonl(args.dev)\\n    random.Random(args.seed).shuffle(dev)\\n    dev = dev[: args.dev_size]\\n    train_cross_encoder(\\n        args.base, [(r[\\\"premise\\\"], r[\\\"hypothesis\\\"]) for r in train], [r[\\\"label\\\"] for r in train], args.out,\\n        dev_auc(dev), epochs=args.epochs, batch_size=args.batch_size, lr=args.lr,\\n        max_length=args.max_length, seed=args.seed, fp16=not args.no_fp16)\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    main()\\n\", \"ml/verifier/evaluate.py\": \"\\\"\\\"\\\"\\nml/verifier/evaluate.py \\u2014 the verifier results table on the held-out RAGTruth test claims.\\n\\n    python -m ml.verifier.evaluate --model models/verifier-v1                 # GPU/CPU with torch\\n    python -m ml.verifier.evaluate --lexical-only --llm-sample 150            # no torch; spends <=150 LLM calls\\n\\nTask: flag UNSUPPORTED claims (hallucination detection), so \\\"positive\\\" = unsupported. Systems:\\n    lexical          share of the claim's content words found in the premise (no learning; the floor)\\n    nli-zero-shot    cross-encoder/nli-deberta-v3-small, P(entailment), NOT fine-tuned by us\\n    fine-tuned       our model (--model)\\n    llm-judge        DeepSeek judging \\\"is this claim supported by the passage\\\" (--llm-sample N rows only)\\nThresholds are chosen on the DEV file (max F1) and applied unchanged to TEST. Reported: precision, recall, F1\\nof the unsupported class, ROC-AUC, ECE of P(supported) (scored systems), latency per claim.\\nWrites ml/data/out/verifier_results.{json,md}.\\n\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport json\\nimport math\\nimport random\\nimport re\\nimport time\\nfrom pathlib import Path\\n\\nfrom ml.eval.classification import best_threshold, ece, prf, roc_auc\\nfrom ml.verifier.sentences import _terms\\n\\nOUT = Path(__file__).parent.parent / \\\"data\\\" / \\\"out\\\"\\nJUDGE_SYSTEM = (\\\"You check whether a CLAIM is fully supported by a PASSAGE. Answer with exactly one word: \\\"\\n                \\\"SUPPORTED if the passage states or clearly implies the claim, otherwise UNSUPPORTED.\\\")\\n\\n\\ndef read(name: str) -> list[dict]:\\n    return [json.loads(x) for x in (OUT / f\\\"verifier_{name}.jsonl\\\").read_text(encoding=\\\"utf-8\\\").splitlines() if x.strip()]\\n\\n\\ndef lexical_scores(rows: list[dict]) -> list[float]:\\n    out = []\\n    for r in rows:\\n        want = _terms(r[\\\"hypothesis\\\"])\\n        out.append(len(want & _terms(r[\\\"premise\\\"])) / len(want) if want else 0.0)\\n    return out\\n\\n\\ndef sigmoid(x: float) -> float:\\n    return 1 / (1 + math.exp(-x))\\n\\n\\ndef model_scores(model_name: str, rows: list[dict], three_way: bool) -> tuple[list[float], float]:\\n    \\\"\\\"\\\"P(supported) per row and mean ms per claim.\\\"\\\"\\\"\\n    from sentence_transformers import CrossEncoder\\n\\n    ce = CrossEncoder(model_name, max_length=384)\\n    pairs = [(r[\\\"premise\\\"], r[\\\"hypothesis\\\"]) for r in rows]\\n    t0 = time.perf_counter()\\n    raw = ce.predict(pairs, batch_size=32, show_progress_bar=False)\\n    ms = 1000 * (time.perf_counter() - t0) / max(1, len(pairs))\\n    if three_way:     # cross-encoder/nli-*: [contradiction, entailment, neutral]\\n        probs = []\\n        for row in raw:\\n            m = max(row)\\n            e = [math.exp(v - m) for v in row]\\n            probs.append(e[1] / sum(e))\\n        return probs, ms\\n    return [sigmoid(float(v)) for v in raw], ms\\n\\n\\ndef score_row(name: str, labels_unsup: list[int], p_supported: list[float], thr_supported: float, ms: float) -> dict:\\n    unsup_score = [1 - p for p in p_supported]\\n    preds = [1 if p < thr_supported else 0 for p in p_supported]\\n    m = prf(labels_unsup, preds)\\n    return {\\\"system\\\": name, \\\"precision\\\": m[\\\"precision\\\"], \\\"recall\\\": m[\\\"recall\\\"], \\\"f1\\\": m[\\\"f1\\\"],\\n            \\\"auc\\\": roc_auc(labels_unsup, unsup_score), \\\"ece\\\": ece([1 - y for y in labels_unsup], p_supported),\\n            \\\"latency_ms\\\": round(ms, 1), \\\"claims\\\": len(labels_unsup)}\\n\\n\\ndef threshold_from_dev(dev_rows: list[dict], dev_p: list[float]) -> float:\\n    unsup = [1 - r[\\\"label\\\"] for r in dev_rows]\\n    t = best_threshold(unsup, [1 - p for p in dev_p])     # threshold on P(unsupported)\\n    return 1 - t\\n\\n\\ndef llm_judge(rows: list[dict]) -> tuple[list[int], float]:\\n    from ml import llm\\n\\n    preds, t0 = [], time.perf_counter()\\n    for r in rows:\\n        ans = llm.chat(f\\\"PASSAGE:\\\\n{r['premise']}\\\\n\\\\nCLAIM:\\\\n{r['hypothesis']}\\\", system=JUDGE_SYSTEM, max_tokens=4, temperature=0)\\n        preds.append(0 if re.match(r\\\"\\\\s*SUPPORTED\\\", ans, re.I) else 1)       # 1 = flagged unsupported\\n    return preds, 1000 * (time.perf_counter() - t0) / max(1, len(rows))\\n\\n\\ndef main() -> None:\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\\"--model\\\", help=\\\"fine-tuned verifier path/id\\\")\\n    ap.add_argument(\\\"--zero-shot\\\", default=\\\"cross-encoder/nli-deberta-v3-small\\\")\\n    ap.add_argument(\\\"--lexical-only\\\", action=\\\"store_true\\\", help=\\\"skip every neural model (no torch needed)\\\")\\n    ap.add_argument(\\\"--llm-sample\\\", type=int, default=0, help=\\\"also run the LLM judge on N test claims\\\")\\n    ap.add_argument(\\\"--seed\\\", type=int, default=13)\\n    args = ap.parse_args()\\n\\n    dev, test = read(\\\"dev\\\"), read(\\\"test\\\")\\n    unsup = [1 - r[\\\"label\\\"] for r in test]\\n    rows: list[dict] = []\\n\\n    dev_lex, test_lex = lexical_scores(dev), lexical_scores(test)\\n    thresholds = {\\\"lexical-overlap\\\": threshold_from_dev(dev, dev_lex)}\\n    rows.append(score_row(\\\"lexical-overlap\\\", unsup, test_lex, thresholds[\\\"lexical-overlap\\\"], 0.0))\\n    scored: dict[str, tuple[list[float], float]] = {\\\"lexical-overlap\\\": (test_lex, 0.0)}\\n\\n    if not args.lexical_only:\\n        for name, path, three in ((\\\"nli-zero-shot\\\", args.zero_shot, True), (\\\"fine-tuned (ours)\\\", args.model, False)):\\n            if not path:\\n                continue\\n            dev_p, _ = model_scores(path, dev, three)\\n            test_p, ms = model_scores(path, test, three)\\n            thresholds[name] = threshold_from_dev(dev, dev_p)\\n            rows.append(score_row(name, unsup, test_p, thresholds[name], ms))\\n            scored[name] = (test_p, ms)\\n\\n    sample_rows: list[dict] = []\\n    if args.llm_sample:\\n        idx = list(range(len(test)))\\n        random.Random(args.seed).shuffle(idx)\\n        idx = sorted(idx[: args.llm_sample])\\n        sub, sub_unsup = [test[i] for i in idx], [unsup[i] for i in idx]\\n        preds, ms = llm_judge(sub)\\n        m = prf(sub_unsup, preds)\\n        sample_rows.append({\\\"system\\\": \\\"llm-judge (DeepSeek)\\\", \\\"precision\\\": m[\\\"precision\\\"], \\\"recall\\\": m[\\\"recall\\\"],\\n                            \\\"f1\\\": m[\\\"f1\\\"], \\\"auc\\\": float(\\\"nan\\\"), \\\"ece\\\": float(\\\"nan\\\"), \\\"latency_ms\\\": round(ms, 1),\\n                            \\\"claims\\\": len(sub)})\\n        for name, (probs, lat) in scored.items():\\n            sample_rows.append(score_row(f\\\"{name} [same subset]\\\", sub_unsup, [probs[i] for i in idx], thresholds[name], lat))\\n\\n    base_rate = sum(unsup) / len(unsup)\\n    cols = [\\\"precision\\\", \\\"recall\\\", \\\"f1\\\", \\\"auc\\\", \\\"ece\\\", \\\"latency_ms\\\", \\\"claims\\\"]\\n\\n    def table(title: str, rs: list[dict]) -> list[str]:\\n        out = [f\\\"### {title}\\\", \\\"\\\", \\\"| system | \\\" + \\\" | \\\".join(cols) + \\\" |\\\", \\\"|---|\\\" + \\\"---|\\\" * len(cols)]\\n        for r in rs:\\n            out.append(f\\\"| {r['system']} | \\\" + \\\" | \\\".join(\\n                \\\"n/a\\\" if (isinstance(r[c], float) and math.isnan(r[c])) else (f\\\"{r[c]:.3f}\\\" if c not in (\\\"latency_ms\\\", \\\"claims\\\") else f\\\"{r[c]:.0f}\\\")\\n                for c in cols) + \\\" |\\\")\\n        return out + [\\\"\\\"]\\n\\n    md = table(f\\\"Held-out RAGTruth test claims (unsupported = positive class; base rate {base_rate:.1%})\\\", rows)\\n    if sample_rows:\\n        md += table(f\\\"LLM-judge comparison on a random subset of {args.llm_sample} test claims\\\", sample_rows)\\n    OUT.mkdir(parents=True, exist_ok=True)\\n    (OUT / \\\"verifier_results.md\\\").write_text(\\\"\\\\n\\\".join(md), encoding=\\\"utf-8\\\")\\n    (OUT / \\\"verifier_results.json\\\").write_text(json.dumps({\\\"test\\\": rows, \\\"subset\\\": sample_rows}, indent=2), encoding=\\\"utf-8\\\")\\n    print(\\\"\\\\n\\\".join(md))\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    main()\\n\"}")
for rel, text in FILES.items():
    p = Path('/kaggle/working/ai') / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(text, encoding='utf-8')
os.chdir('/kaggle/working/ai')
print(len(FILES), 'files written')

## 2. Build datasets (public data; rate-limited, takes a while)

In [ ]:
!python -m ml.data.make_reranker_data

In [ ]:
!python -m ml.verifier.make_data --mnli-rows 6000

## 3. Train the reranker (M1)

In [ ]:
!python -m ml.train_reranker --train ml/data/out/train.jsonl --dev ml/data/out/dev.jsonl --base cross-encoder/ms-marco-MiniLM-L-6-v2 --out models/reranker-v1 --epochs 2

## 4. Retrieval ablation on held-out SciFact test queries

In [ ]:
!python -m ml.eval.run_retrieval_eval --dense BAAI/bge-small-en-v1.5 --rerank cross-encoder/ms-marco-MiniLM-L-6-v2 --rerank models/reranker-v1

## 5. Train the verifier (M2)

In [ ]:
!python -m ml.verifier.train_verifier --train ml/data/out/verifier_train.jsonl --dev ml/data/out/verifier_dev.jsonl --out models/verifier-v1 --epochs 2

## 6. Verifier evaluation on held-out RAGTruth test claims

In [ ]:
!python -m ml.verifier.evaluate --model models/verifier-v1

## 7. Package results and models for download

In [ ]:
import shutil
shutil.copy('ml/data/out/retrieval_results.md', '/kaggle/working/')
shutil.copy('ml/data/out/retrieval_results.json', '/kaggle/working/')
shutil.copy('ml/data/out/verifier_results.md', '/kaggle/working/')
shutil.copy('ml/data/out/verifier_results.json', '/kaggle/working/')
shutil.make_archive('/kaggle/working/collabmind_models', 'zip', 'models')
print(open('/kaggle/working/retrieval_results.md').read())
print(open('/kaggle/working/verifier_results.md').read())